# 테이블 적재

탐색 단계에서는 매 쿼리마다 `read_csv_auto('경로')` 로 CSV 를 다시 파싱.
테이블이 늘고 조인이 섞이면 쿼리에서 경로가 차지하는 비중이 커져 로직이 묻히고,
100 만 행짜리 `geolocation` 은 조회할 때마다 파싱 비용이 재발생.

9 개 CSV 를 DuckDB 파일에 영구 테이블로 적재해 이후 분석이 `FROM orders` 만으로 시작하도록 구성.

**적재 기준**

- 저장소: `olist.duckdb` (프로젝트 루트, `.gitignore` 대상)
- 테이블명: 파일명에서 `olist_` 접두사와 `_dataset` 접미사를 제거
- 타입: `read_csv_auto` 의 추론 결과를 그대로 사용
- 원본 CSV 는 수정하지 않으며, 적재는 언제든 재실행 가능

## 1. 연결

인메모리 연결은 커널이 종료되면 테이블도 함께 소멸.
적재 결과를 유지하려면 파일 경로를 지정한 연결이 필요.

In [ ]:
import duckdb

con = duckdb.connect('../olist.duckdb')

## 2. 대상 파일과 테이블명 매핑

적재 전에 파일명 → 테이블명 변환 결과를 확인.
`product_category_name_translation` 은 다른 파일과 명명 규칙이 달라
접두사·접미사 제거가 적용되지 않음.

In [4]:
import glob
import os

csv_files = sorted(glob.glob("../archive/*.csv"))

tables = []

for path in csv_files:
    file = os.path.basename(path)

    name = file
    name = name.replace("olist_", "")
    name = name.replace("_dataset", "")
    name = name.replace(".csv", "")

    tables.append((name, path))
    print(f"{name:35s} <- {file}")

customers                           <- olist_customers_dataset.csv
geolocation                         <- olist_geolocation_dataset.csv
order_items                         <- olist_order_items_dataset.csv
order_payments                      <- olist_order_payments_dataset.csv
order_reviews                       <- olist_order_reviews_dataset.csv
orders                              <- olist_orders_dataset.csv
products                            <- olist_products_dataset.csv
sellers                             <- olist_sellers_dataset.csv
product_category_name_translation   <- product_category_name_translation.csv


## 3. 단건 적재 검증 — orders

9 개를 한 번에 돌리기 전에 `orders` 하나로 적재문을 검증.
적재된 행 수를 원본 CSV 와 대조.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE orders AS
SELECT * 
FROM read_csv_auto('../archive/olist_orders_dataset.csv')
""")

In [8]:
con.execute("""
SELECT COUNT(*) AS order_cnt
FROM orders
""").df()

,order_cnt
0,99441


## 4. 전체 적재

검증된 적재문을 9 개 파일에 반복 적용.

`CREATE OR REPLACE TABLE` 로 노트북을 여러 번 실행해도 결과가 동일하도록 구성.
`DROP` 후 `CREATE` 하는 방식과 달리 교체가 성공했을 때만 반영되므로,
적재가 중간에 실패해도 기존 테이블이 잔존.

테이블명은 SQL 파라미터로 바인딩할 수 없어 f-string 으로 조립.
치환되는 값은 외부 입력이 아니라 `archive/` 의 파일명에서 생성한 값.

In [ ]:
for name, path in tables:

    sql = f"""
    CREATE OR REPLACE TABLE {name} AS
    SELECT *
    FROM read_csv_auto('{path}')
    """

    con.execute(sql)
    print(f"완료: {name}")

완료: customers
완료: geolocation
완료: order_items
완료: order_payments
완료: order_reviews
완료: orders
완료: products
완료: sellers
완료: product_category_name_translation


## 5. 적재 검증

적재 과정에서의 행 누락·중복 여부를 원본 CSV 행 수와 대조.

In [16]:
con.execute("""
SELECT table_name
FROM information_schema.tables
""").df()

,table_name
0,customers
1,geolocation
2,orders
3,order_items
4,order_payments
5,order_reviews
6,products
7,product_category_name_translation
8,sellers


In [17]:
for name, path in tables:

    rows = con.execute(
        f"SELECT COUNT(*) FROM {name}"
    ).fetchone()[0]

    csv_rows = con.execute(
        f"SELECT COUNT(*) FROM read_csv_auto('{path}')"
    ).fetchone()[0]

    mark = "OK" if rows == csv_rows else "불일치"

    print(f"{name:35s} {rows:>9,} / {csv_rows:>9,}  {mark}")

customers                              99,441 /    99,441  OK
geolocation                         1,000,163 / 1,000,163  OK
order_items                           112,650 /   112,650  OK
order_payments                        103,886 /   103,886  OK
order_reviews                          99,224 /    99,224  OK
orders                                 99,441 /    99,441  OK
products                               32,951 /    32,951  OK
sellers                                 3,095 /     3,095  OK
product_category_name_translation          71 /        71  OK


## 6. 적재 결과 확인

경로 없이 테이블명만으로 조인이 가능한지, 주(州)별 주문 건수 상위 5 개로 확인.

In [ ]:
con.execute("""
SELECT c.customer_state,
       COUNT(*) AS cs_cnt
FROM orders AS o
JOIN customers AS c
  ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY cs_cnt DESC
LIMIT 5
""").df()

,customer_state,cs_cnt
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045


## 7. 연결 종료

DuckDB 파일은 한 프로세스만 쓰기 모드로 열 수 있으므로 다음 노트북을 위해 여기서 종료.

In [ ]:
con.close()